In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

# 1. L'architecture (doit être identique à l'entraînement)
class DinoClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.classifier = nn.Linear(384, num_classes)
    def forward(self, x):
        return self.classifier(self.backbone(x))

# 2. Configuration & Chargement
classes = ['00-normal', '01-minor', '02-moderate', '03-severe']
device = "cpu"

model = DinoClassifier(num_classes=4).to(device)
model.load_state_dict(torch.load('best_dino_finetuned.pth', map_location=device))
model.eval()

# 3. Préparation de l'image & Prédiction
transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

sample_image_path = "C:/Users/huber_otpq54a/OneDrive/Documents/Formation/IA/Developpement/Projets/Deep_Learning/Test/accident-de-voiture.webp"
img = transform(Image.open(sample_image_path).convert('RGB')).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(img)
    # On transforme TOUS les scores en probabilités (0.0 à 1.0)
    probabilities = torch.nn.functional.softmax(output, dim=1)[0]

probs_triées, indices = torch.sort(probabilities, descending=True)

for i in range(len(indices)):
    print(f"{classes[indices[i]]:<15} : {probs_triées[i].item()*100:>6.2f}%")

# Optionnel : Afficher directement le gagnant
index_max = probabilities.argmax().item()
print(f"\nPrédiction finale : {classes[index_max]}")

Using cache found in C:\Users\huber_otpq54a/.cache\torch\hub\facebookresearch_dinov2_main


02-moderate     :  57.33%
03-severe       :  39.57%
01-minor        :   3.09%
00-normal       :   0.01%

Prédiction finale : 02-moderate
